In [9]:
import os

# Check current working directory
print("Current directory:", os.getcwd())

# Check Downloads folder
downloads = os.path.expanduser("~/Downloads")
print("\nDownloads folder contents:")
for f in os.listdir(downloads):
    if "JACOB" in f.upper():
        print(f)

Current directory: C:\Users\mehvish shaikh\OneDrive\Documents\Thesis

Downloads folder contents:


In [10]:
import os

# Check current Thesis folder for it too
print("In Thesis folder:")
for f in os.listdir("."):
    if "JACOB" in f.upper():
        print(f)

# Check Downloads for any zip at all (in case it's named differently)
downloads = os.path.expanduser("~/Downloads")
print("\nAll files in Downloads:")
print(os.listdir(downloads))

In Thesis folder:
borenstein-lab microbiome-metabolome-curated-data main data-processed_data_JACOBS_IBD_FAMILIES_2016
borenstein-lab microbiome-metabolome-curated-data main data-processed_data_JACOBS_IBD_FAMILIES_2016.zip

All files in Downloads:
[' CAPSTONE1_PROJECT.ipynb', '.ipynb_checkpoints', '1-s2.0-S0925443924006124-main.docx', '11306_2019_Article_1612.pdf', '125_2025_Article_6419.pdf', '12916_2022_Article_2551.pdf', '12967_2025_Article_6720.pdf', '13300_2024_Article_1676.pdf', '16_weeks_plan_thesis_metabolite.pptx', '1777912651.8878841.MOV', '3300460.nbib', '41586_2019_Article_1237.pdf', '41591_2023_Article_2640.pdf', '41598_2017_Article_10034.pdf', 'ABSOLUTE_FINAL_mz_RT_HMDB_KEGG.xls', 'acp_aim107_224.bib', 'Advancing AI for multi-omics and clinical data integration in basic and translational cancer research.pdf', 'AI PS CV.pdf', 'Anaconda3-2025.06-0-Windows-x86_64.exe', 'Artificial intelligence for the prevention and clinical management.pdf', 'Assessment_Presentation_2026.pptx

In [11]:
import os

# Find the exact folder name (since it has spaces in it)
matches = [f for f in os.listdir(".") if "JACOBS" in f.upper() and not f.endswith(".zip")]
print("Matching folder(s):", matches)

if matches:
    folder_name = matches[0]
    print("\nContents of that folder:")
    print(os.listdir(folder_name))

Matching folder(s): ['borenstein-lab microbiome-metabolome-curated-data main data-processed_data_JACOBS_IBD_FAMILIES_2016']

Contents of that folder:
['.RData', 'genera.counts.tsv', 'genera.tsv', 'metadata.tsv', 'mtb.map.tsv', 'mtb.tsv']


In [12]:
import pandas as pd

folder_name = "borenstein-lab microbiome-metabolome-curated-data main data-processed_data_JACOBS_IBD_FAMILIES_2016"

metadata = pd.read_csv(f"{folder_name}/metadata.tsv", sep="\t")
mtb = pd.read_csv(f"{folder_name}/mtb.tsv", sep="\t")

print("Metadata shape:", metadata.shape)
print("Metabolite table shape:", mtb.shape)
print(metadata.columns.tolist())
print(metadata["Study.Group"].value_counts())

Metadata shape: (90, 11)
Metabolite table shape: (90, 4627)
['Dataset', 'Sample', 'Subject', 'Study.Group', 'Age', 'Age.Units', 'Gender', 'DOI', 'Publication.Name', 'Family_ID', 'Pedigree']
Study.Group
Normal    54
CD        26
UC        10
Name: count, dtype: int64


In [13]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder

feature_cols = [c for c in mtb.columns if c != "Sample"]
X = metadata.merge(mtb, on="Sample")[feature_cols]
y = metadata["Study.Group"]

print("X shape:", X.shape)
print("y distribution:", y.value_counts())

X shape: (90, 4626)
y distribution: Study.Group
Normal    54
CD        26
UC        10
Name: count, dtype: int64


In [15]:
mtb_map = pd.read_csv(f"{folder_name}/mtb.map.tsv", sep="\t")
print(mtb_map.columns.tolist())
print(mtb_map.head())

['Compound', 'Compound.Name', 'ESI.mode', 'm.z', 'Retention.time', 'Putative.ID.that.did.not.validate', 'HMDB.identification', 'KEGG.identification', 'LIPIDMAPS.identification', 'BioCyc.identification', 'HMDB', 'KEGG', 'High.Confidence.Annotation']
                   Compound                      Compound.Name  ESI.mode  \
0  Negative_515.2282_5.6026  11-Oxo-androsterone [glucuronide]  Negative   
1   Positive_371.2584_5.555        3-Oxo-4,6-choladienoic acid  Positive   
2  Positive_265.1758_0.7696            3-Oxotetradecanoic acid  Positive   
3  Negative_493.2234_5.8191            3-Sulfodeoxycholic acid  Negative   
4  Positive_154.0509_0.9254              5-aminosalicylic acid  Positive   

        m.z  Retention.time Putative.ID.that.did.not.validate  \
0  515.2282          5.6026                               NaN   
1  371.2584          5.5550                               NaN   
2  265.1758          0.7696                               NaN   
3  493.2234          5.8191       

In [16]:
# Animesh's rule: remove if NO HMDB AND NO KEGG AND NO Compound Name
unknown_mask = mtb_map["HMDB"].isna() & mtb_map["KEGG"].isna() & mtb_map["Compound.Name"].isna()
unknown_compounds = mtb_map.loc[unknown_mask, "Compound"].tolist()

print("Total metabolites in map:", len(mtb_map))
print("Unknown (no ID at all):", len(unknown_compounds))

# Apply to our already sparsity-filtered X_filtered
cols_to_keep = [c for c in X_filtered.columns if c not in unknown_compounds]
X_known = X_filtered[cols_to_keep]

print("Features after removing unknowns:", X_known.shape[1], "of", X_filtered.shape[1])

Total metabolites in map: 4626
Unknown (no ID at all): 4580
Features after removing unknowns: 45 of 2342


In [17]:
X_known_clr = clr_transform(X_known)

# Redo the train/test split on this reduced set
X_train2 = X_known_clr.iloc[train_idx]
X_test2 = X_known_clr.iloc[test_idx]

scaler2 = StandardScaler()
X_train2_scaled = pd.DataFrame(scaler2.fit_transform(X_train2), columns=X_train2.columns)
X_test2_scaled = pd.DataFrame(scaler2.transform(X_test2), columns=X_test2.columns)

print("Train shape:", X_train2_scaled.shape)
print("Test shape:", X_test2_scaled.shape)

Train shape: (72, 45)
Test shape: (18, 45)


In [18]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=0.05, random_state=42)
outlier_flags = iso.fit_predict(X_known_clr)

print("Outliers detected:", (outlier_flags == -1).sum(), "of", len(X_known_clr))

Outliers detected: 5 of 90


In [19]:
# Remove the 5 outlier patients
X_clean = X_known_clr[outlier_flags == 1].reset_index(drop=True)
y_clean = y.reset_index(drop=True)[outlier_flags == 1].reset_index(drop=True)
metadata_clean = metadata.reset_index(drop=True)[outlier_flags == 1].reset_index(drop=True)

print("Samples after outlier removal:", len(X_clean), "of", len(X_known_clr))
print(y_clean.value_counts())

Samples after outlier removal: 85 of 90
Study.Group
Normal    52
CD        23
UC        10
Name: count, dtype: int64


In [20]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Binary framing (IBD vs Normal), matching Animesh's original approach
y_binary_clean = y_clean.map(lambda g: "Normal" if g == "Normal" else "IBD")

ibd_mask = (y_binary_clean == "IBD").values
normal_mask = (y_binary_clean == "Normal").values

pvals = {}
for col in X_clean.columns:
    try:
        stat, p = mannwhitneyu(X_clean.loc[ibd_mask, col], X_clean.loc[normal_mask, col])
    except ValueError:
        p = 1.0
    pvals[col] = p

pval_series = pd.Series(pvals)
fdr_corrected = multipletests(pval_series.values, method="fdr_bh")[1]
fdr_series = pd.Series(fdr_corrected, index=pval_series.index)

significant_features = fdr_series[fdr_series < 0.05].index.tolist()
print("Significant metabolites at FDR < 0.05:", len(significant_features), "of", len(X_clean.columns))
print(significant_features)

Significant metabolites at FDR < 0.05: 27 of 45
['Positive_116.0709_0.8555', 'Positive_132.1024_1.1392', 'Positive_144.0814_2.9765', 'Positive_145.1329_1.0178', 'Positive_145.1336_0.8918', 'Positive_175.1091_1.0267', 'Positive_182.0813_0.9824', 'Positive_261.1444_1.0266', 'Positive_295.1654_2.8735', 'Positive_309.1775_3.0874', 'Positive_355.2629_5.1581', 'Positive_407.2797_5.5815', 'Positive_431.2786_5.9801', 'Negative_124.0067_0.8872', 'Negative_128.0347_0.9594', 'Negative_146.0453_0.8921', 'Negative_151.0255_0.9921', 'Negative_180.0642_1.0351', 'Negative_187.0971_4.3188', 'Negative_188.0559_1.0259', 'Negative_319.166_4.5962', 'Negative_405.2632_5.6659', 'Negative_471.2406_5.2037', 'Negative_471.2406_5.8057', 'Negative_493.2234_5.8191', 'Negative_591.3178_4.6332', 'Negative_593.3325_4.6412']


In [22]:
from sklearn.metrics import roc_auc_score

In [23]:
rf_auc_f = roc_auc_score(y_test_f, rf_probs_f, multi_class="ovr", average="macro")
xgb_auc_f = roc_auc_score(y_test_f, xgb_probs_f, multi_class="ovr", average="macro")

print(f"Random Forest — macro-AUC: {rf_auc_f:.3f}")
print(f"XGBoost — macro-AUC: {xgb_auc_f:.3f}")
print(f"Classes: {le_final.classes_}")
print(f"Test set size: {len(y_test_f)}")

Random Forest — macro-AUC: 0.693
XGBoost — macro-AUC: 0.731
Classes: ['CD' 'Normal' 'UC']
Test set size: 17


In [24]:
X_final = X_clean[significant_features]
le_final = LabelEncoder()
y_final_enc = le_final.fit_transform(y_clean)

train_idx2, test_idx2 = train_test_split(np.arange(len(y_final_enc)), test_size=0.2, stratify=y_final_enc, random_state=42)
y_train_f, y_test_f = y_final_enc[train_idx2], y_final_enc[test_idx2]

scaler_final = StandardScaler()
X_train_f = pd.DataFrame(scaler_final.fit_transform(X_final.iloc[train_idx2]), columns=X_final.columns)
X_test_f = pd.DataFrame(scaler_final.transform(X_final.iloc[test_idx2]), columns=X_final.columns)

rf_final = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)
rf_final.fit(X_train_f, y_train_f)
rf_probs_f = rf_final.predict_proba(X_test_f)

xgb_final = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, eval_metric="mlogloss")
xgb_final.fit(X_train_f, y_train_f)
xgb_probs_f = xgb_final.predict_proba(X_test_f)

rf_auc_f = roc_auc_score(y_test_f, rf_probs_f, multi_class="ovr", average="macro")
xgb_auc_f = roc_auc_score(y_test_f, xgb_probs_f, multi_class="ovr", average="macro")

print(f"Random Forest — macro-AUC: {rf_auc_f:.3f}")
print(f"XGBoost — macro-AUC: {xgb_auc_f:.3f}")
print(f"Classes: {le_final.classes_}")
print(f"Test set size: {len(y_test_f)}")

Random Forest — macro-AUC: 0.693
XGBoost — macro-AUC: 0.731
Classes: ['CD' 'Normal' 'UC']
Test set size: 17
